# Small Morton set VAE (low VRAM)

Train a **tiny conditional set VAE** on on-the-fly Morton subsets while the
GPU BH corpus is still writing dumps. Defaults stay on **CPU** with
`N=64`, `d_model=64`, `batch_size=1` so evolve jobs keep the GPU.

Docs: [`morton_generative.md`](../docs/morton_generative.md).

## Prerequisites

```bash
pip install -e '.[ml]'
# Optional: build a manifest over whatever dumps exist so far
galacticsics-morton-index runs/mw_morton_corpus_v2 \
  -o runs/mw_morton_corpus_v2/snapshot_manifest.json
```

If no corpus is ready yet, the notebook synthesises a few fake snapshots.

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader

# Low-VRAM / share-GPU-with-corpus defaults
DEVICE = "cpu"  # use "cuda" only if the evolve campaign is idle
N_PARTICLES = 64
BATCH_SIZE = 1
D_MODEL = 64
LATENT_DIM = 32
EPOCHS = 3
LR = 1e-3
SEED = 0

REPO = Path("..").resolve()
MANIFEST_CANDIDATES = [
    REPO / "runs/mw_morton_corpus_v2/snapshot_manifest.json",
    REPO / "runs/mw_morton_corpus/snapshot_manifest.json",
]
OUT = REPO / "notebooks/artifacts/morton_vae_small"
OUT.mkdir(parents=True, exist_ok=True)
print(f"torch {torch.__version__}  device={DEVICE}")

In [ ]:
import json

from galacticsics.ml.morton.dataset import MortonSnapshotDataset, collate_morton_batch
from galacticsics.ml.morton.index import SnapshotRecord, write_snapshot_manifest
from galacticsics.ml.models.sequence_vae import SequenceVAE, SequenceVAEConfig


def _synthetic_manifest(root: Path, n_runs: int = 4) -> Path:
    """Tiny fake corpus so the notebook runs without campaign dumps."""
    root.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(SEED)
    for i in range(n_runs):
        run = root / f"synth{i:04d}"
        run.mkdir(exist_ok=True)
        (run / "model.json").write_text(
            json.dumps(
                {
                    "label": f"synth{i}",
                    "disk": {"mass": 10.0 + i, "scale_length": 2.5, "scale_height": 0.3, "enabled": True},
                    "halo": {"v0": 3.5, "a": 30.0, "enabled": True},
                    "bulge": {"v0": 1.5, "a": 0.6, "enabled": True},
                    "disk_kinematics": {"toomre_q_target": 1.5, "sigma_r0": 0.5},
                }
            )
        )
        n = 400
        pos = rng.normal(scale=[4, 4, 0.4], size=(n, 3))
        vel = rng.normal(scale=0.2, size=(n, 3))
        type_id = np.array([0] * 220 + [1] * 120 + [2] * 60, dtype=np.int32)
        np.savez(
            run / "ic_state.npz",
            pos=pos,
            vel=vel,
            mass=np.ones(n) / n,
            eps=np.full(n, 0.1),
            type_id=type_id,
        )
    return write_snapshot_manifest(root)


manifest = next((p for p in MANIFEST_CANDIDATES if p.is_file()), None)
if manifest is None:
    manifest = _synthetic_manifest(OUT / "synthetic_corpus")
    print(f"no campaign manifest; using synthetic {manifest}")
else:
    print(f"using {manifest}")

ds = MortonSnapshotDataset(
    manifest,
    n_particles=N_PARTICLES,
    split="train",
    seed=SEED,
)
if len(ds) == 0:
    # Hash split may put all synth runs in val/test; reopen without filter
    ds = MortonSnapshotDataset(manifest, n_particles=N_PARTICLES, split=None, seed=SEED)
print(f"snapshots={len(ds)}  theta_dim={len(ds.theta_keys)}  N={N_PARTICLES}")

In [ ]:
def collate(batch):
    np_batch = collate_morton_batch(batch)
    return {k: torch.as_tensor(np_batch[k]) for k in ("c", "dm", "dx", "v", "theta")}


loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
cfg = SequenceVAEConfig(
    n_particles=N_PARTICLES,
    theta_dim=len(ds.theta_keys),
    d_model=D_MODEL,
    latent_dim=LATENT_DIM,
    n_layers=1,
    n_heads=2,
    n_decode_layers=1,
)
model = SequenceVAE(cfg).to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=LR)
n_params = sum(p.numel() for p in model.parameters())
print(f"params={n_params:,}")

In [ ]:
history = []
model.train()
for epoch in range(EPOCHS):
    losses = []
    last = {}
    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        out = model(batch["c"], batch["dm"], batch["dx"], batch["v"], batch["theta"])
        metrics = model.loss(batch, out)
        opt.zero_grad(set_to_none=True)
        metrics["loss"].backward()
        opt.step()
        losses.append(float(metrics["loss"].detach().cpu()))
        last = {k: float(v.detach().cpu()) for k, v in metrics.items()}
    mean_loss = float(np.mean(losses))
    history.append(mean_loss)
    print(
        f"epoch {epoch + 1}/{EPOCHS}  loss={mean_loss:.4f}  "
        f"recon={last.get('recon', float('nan')):.3f}  "
        f"ce={last.get('ce', float('nan')):.3f}  "
        f"kl={last.get('kl', float('nan')):.3f}  "
        f"profile={last.get('profile', float('nan')):.3f}"
    )

ckpt = OUT / "sequence_vae.pt"
torch.save({"model": model.state_dict(), "config": cfg.__dict__, "theta_keys": ds.theta_keys}, ckpt)
print(f"wrote {ckpt}")

In [ ]:
import matplotlib.pyplot as plt

model.eval()
theta = torch.as_tensor(ds[0]["theta"], dtype=torch.float32, device=DEVICE).unsqueeze(0)
gen = model.generate(theta, n=N_PARTICLES * 2, chunk_size=N_PARTICLES)
pos = gen["dx"][0]
c = gen["c"][0]

fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
axes[0].plot(history, marker="o")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("loss")
axes[0].set_title("train loss")
for cid, name, color in [(0, "disk", "C0"), (1, "halo", "C1"), (2, "bulge", "C2")]:
    m = c == cid
    if np.any(m):
        axes[1].scatter(pos[m, 0], pos[m, 1], s=6, alpha=0.6, label=name, c=color)
axes[1].set_aspect("equal")
axes[1].set_xlabel("x [kpc]")
axes[1].set_ylabel("y [kpc]")
axes[1].set_title("generate (xy)")
axes[1].legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUT / "vae_small_overview.png", dpi=120)
plt.show()
print(f"generated std(pos)={pos.std():.3f} (should be > 0 — set decode uses noise queries)")